# Simulación · Unidad Modular III
## Dinámica de sistemas y aplicaciones

Este cuaderno acompaña la presentación *Unidad3_Dinamica_de_Sistemas.pptx*. Todo se programa desde cero con **NumPy**, **SciPy**, **pandas**, **Matplotlib**, **seaborn** y **NetworkX**, para que veas cómo funciona por dentro un software como Vensim o Stella.

| Parte | Contenido |
|---|---|
| 0 | Preparación: integradores de Euler y Runge-Kutta 4 y un mini-motor de niveles y flujos |
| 1 | Dinámica de sistemas: la bañera, niveles y flujos |
| 2 | Características estructurales y funcionales: invernadero, paso DT, sensibilidad y dominancia de bucles |
| 3 | Trayectorias, equilibrios, retratos de fase, ciclos límite y caos |
| 4 | Bifurcaciones (silla-nodo, transcrítica, horquilla, Hopf, duplicación de periodo) y catástrofes (cúspide, histéresis, alerta temprana) |
| 5 | Diagramas causales como grafos con signo: detección automática de bucles y su polaridad |
| 6 | Simbología: STEP, PULSE, RAMP, DELAY1, DELAY3, SMOOTH y tablas |
| 7 | Etapas para elaborar un modelo: caso completo de una epidemia en el campus |
| 8 | Los 18 ejercicios de la presentación, resueltos |

> Ejecuta en orden con *Entorno de ejecución → Ejecutar todas*.

---
## 0. Preparación

In [ ]:
import math, itertools, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx
from scipy.optimize import brentq
warnings.filterwarnings("ignore")

MORADO, VERDE, AMBAR, CORAL, AZUL, GRIS, TINTA = "#6C4AB6", "#00A896", "#F2A900", "#E4572E", "#3A86C8", "#6B7280", "#1A1433"
PAL = [MORADO, VERDE, CORAL, AZUL, AMBAR, TINTA]
sns.set_theme(style="whitegrid", palette=PAL)
plt.rcParams.update({"figure.figsize": (10, 4.2), "axes.titleweight": "bold", "axes.titlesize": 12})
pd.set_option("display.precision", 4)

### Integradores numéricos

Un modelo de dinámica de sistemas es un sistema de ecuaciones diferenciales $\dot{\mathbf{x}} = f(t, \mathbf{x})$, donde $\mathbf{x}$ son los **niveles**.

* **Euler**: $\mathbf{x}(t+DT) = \mathbf{x}(t) + DT\, f(t, \mathbf{x})$ — lo que hace Vensim por defecto.
* **Runge-Kutta 4**: promedia cuatro pendientes; mucho más exacto con el mismo $DT$.

In [ ]:
def simular(f, x0, T, dt=0.1, metodo="rk4"):
    # Devuelve (t, X) con X de forma (pasos, n_niveles).
    n = int(round(T / dt)); x = np.array(x0, dtype=float); t = 0.0
    ts, xs = [t], [x.copy()]
    for _ in range(n):
        if metodo == "euler":
            x = x + dt * np.asarray(f(t, x), dtype=float)
        else:
            k1 = np.asarray(f(t, x), float); k2 = np.asarray(f(t + dt/2, x + dt/2*k1), float)
            k3 = np.asarray(f(t + dt/2, x + dt/2*k2), float); k4 = np.asarray(f(t + dt, x + dt*k3), float)
            x = x + dt/6 * (k1 + 2*k2 + 2*k3 + k4)
        t += dt; ts.append(t); xs.append(x.copy())
    return np.array(ts), np.array(xs)

### Un mini-motor de niveles y flujos (como Vensim)

Definimos el modelo con diccionarios: **niveles** (valor inicial y lista de flujos que entran/salen), **flujos** y **auxiliares** (funciones de las demás variables). El motor ordena el cálculo en cada paso: auxiliares → flujos → integración de niveles.

In [ ]:
class ModeloDS:
    def __init__(self, niveles, flujos, auxiliares=None, constantes=None):
        # niveles: {nombre: (inicial, [entradas], [salidas])}
        self.niveles, self.flujos = niveles, flujos
        self.aux, self.const = auxiliares or {}, constantes or {}
    def _estado(self, t, x):
        v = dict(self.const); v["t"] = t
        v.update({k: x[i] for i, k in enumerate(self.niveles)})
        for k, fn in self.aux.items(): v[k] = fn(v)          # en orden de definición
        for k, fn in self.flujos.items(): v[k] = fn(v)
        return v
    def derivadas(self, t, x):
        v = self._estado(t, x)
        return [sum(v[e] for e in ent) - sum(v[s] for s in sal) for (_, ent, sal) in self.niveles.values()]
    def correr(self, T, dt=0.125, metodo="euler"):
        x0 = [ini for (ini, _, _) in self.niveles.values()]
        t, X = simular(self.derivadas, x0, T, dt, metodo)
        filas = [self._estado(tt, xx) for tt, xx in zip(t, X)]
        return pd.DataFrame(filas).set_index("t")
    def estructura(self):
        filas = [(k, "Nivel") for k in self.niveles] + [(k, "Flujo") for k in self.flujos] + [(k, "Auxiliar") for k in self.aux] + [(k, "Constante") for k in self.const]
        return pd.DataFrame(filas, columns=["variable", "tipo"])
print("Motor listo")

---
## 1. Dinámica de sistemas: la bañera

El nivel (agua) solo cambia por sus flujos:
$$\frac{d\,\text{Agua}}{dt} = \text{Entrada} - \text{Salida}$$

Experimento: la entrada sube de 5 a 10 L/min en el minuto 10 y baja a 2 L/min en el minuto 25; el desagüe saca un caudal proporcional al agua (salida = agua/τ, τ = 4 min).

In [ ]:
banera = ModeloDS(
    niveles={"agua": (20, ["entrada"], ["salida"])},
    flujos={"entrada": lambda v: 5 + 5*(v["t"] >= 10) - 8*(v["t"] >= 25), "salida": lambda v: v["agua"] / v["tau"]},
    constantes={"tau": 4})
df = banera.correr(40, dt=0.05)
fig, ax = plt.subplots(1, 2, figsize=(14, 4))
df[["entrada", "salida"]].plot(ax=ax[0], title="Flujos (L/min)"); df["agua"].plot(ax=ax[1], color=AZUL, title="Nivel: agua en la bañera (L)")
for a in ax: a.axvline(10, color=GRIS, ls=":"); a.axvline(25, color=GRIS, ls=":")
plt.show()
display(banera.estructura())
print("Equilibrio: agua* = entrada × τ →", 5*4, "L, luego", 10*4, "L y luego", 2*4, "L")

**Observa:** el nivel no salta; se acerca a su nuevo equilibrio con una constante de tiempo τ. Mientras entrada > salida, el agua sube, aunque la entrada esté bajando. Esa memoria es lo que dan los **niveles**.

---
## 2. Características estructurales y funcionales

### 2.1 Estructura de un invernadero

Niveles: humedad del suelo y agua en el tanque. Flujos: riego, evaporación y bombeo de reposición. Auxiliares: brecha de humedad. Constantes: objetivo y tiempo de riego. Entorno: lluvia (exógena). El controlador riega en proporción a la brecha (control proporcional).

In [ ]:
rng = np.random.default_rng(7)
lluvia_diaria = rng.poisson(2.4/7, 200) * 3.0        # exógena: Poisson, 3 % por lluvia
inv = ModeloDS(
    niveles={"humedad": (32, ["riego", "lluvia"], ["evaporacion"]), "tanque": (500, ["reposicion"], ["consumo"])},
    auxiliares={"brecha": lambda v: v["objetivo"] - v["humedad"],
                "lluvia": lambda v: lluvia_diaria[min(int(v["t"]), 199)]},
    flujos={"riego": lambda v: max(0.0, v["brecha"]) / v["t_riego"] * (v["tanque"] > 10), "evaporacion": lambda v: v["k_evap"] * v["humedad"],
            "consumo": lambda v: v["riego"] * v["litros_por_punto"], "reposicion": lambda v: (600 - v["tanque"]) / v["t_repos"]},
    constantes={"objetivo": 34, "t_riego": 2, "k_evap": 0.12, "litros_por_punto": 20, "t_repos": 5})
display(inv.estructura())
df = inv.correr(120, dt=0.05)
fig, ax = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
df["humedad"].plot(ax=ax[0], color=VERDE, title="Humedad del suelo (%)"); ax[0].axhline(34, color=CORAL, ls="--", label="objetivo"); ax[0].legend()
df["tanque"].plot(ax=ax[1], color=AZUL, title="Agua en el tanque (L)"); plt.tight_layout(); plt.show()

El bucle **humedad → brecha → riego → humedad** es de balance (B): mantiene la humedad cerca del objetivo. La evaporación es otro bucle B (más humedad → más evaporación). La lluvia, exógena, solo perturba.

Fíjate en que la humedad se estabiliza **por debajo** del objetivo: el control proporcional deja un error en estado estacionario, porque necesita una brecha para seguir regando y compensar la evaporación. En equilibrio sin lluvia: $H^* = \dfrac{\text{objetivo}/t_{riego}}{k_{evap} + 1/t_{riego}} = \dfrac{17}{0.62} \approx 27.4$ %.

### 2.2 Característica funcional: elegir el paso DT

Regla: $DT \le \tfrac14$ de la constante de tiempo más pequeña. Comparamos Euler con distintos DT frente a la solución exacta del crecimiento exponencial.

In [ ]:
f = lambda t, x: [0.06 * x[0]]
filas = []
fig, ax = plt.subplots()
tt = np.linspace(0, 50, 200); ax.plot(tt, 10*np.exp(0.06*tt), color=TINTA, lw=2.5, label="exacta")
for dt, c in [(5, CORAL), (2, AMBAR), (1, VERDE), (0.25, AZUL)]:
    t, X = simular(f, [10], 50, dt, "euler"); ax.plot(t, X[:, 0], "o-", ms=3, color=c, label=f"Euler DT={dt}")
    filas.append(dict(DT=dt, metodo="Euler", x50=X[-1, 0], error_pct=100*(X[-1, 0] - 10*np.exp(3))/(10*np.exp(3))))
    t, X = simular(f, [10], 50, dt, "rk4"); filas.append(dict(DT=dt, metodo="RK4", x50=X[-1, 0], error_pct=100*(X[-1, 0] - 10*np.exp(3))/(10*np.exp(3))))
ax.legend(); ax.set_title("Euler con distintos DT"); plt.show()
pd.DataFrame(filas).round(4)

### 2.3 Sensibilidad selectiva

Pocos parámetros mueven mucho el resultado. Variamos ±20 % cada parámetro del invernadero y medimos el cambio en la humedad media.

In [ ]:
base = inv.correr(120, dt=0.05)["humedad"].mean()
filas = []
for p in ["objetivo", "t_riego", "k_evap", "litros_por_punto", "t_repos"]:
    for f_ in (0.8, 1.2):
        m = ModeloDS(inv.niveles, inv.flujos, inv.aux, {**inv.const, p: inv.const[p] * f_})
        filas.append(dict(parametro=p, cambio=f"{round((f_-1)*100):+d} %", humedad_media=m.correr(120, dt=0.05)["humedad"].mean()))
sens = pd.DataFrame(filas); sens["efecto_%"] = 100 * (sens.humedad_media - base) / base
display(sens.round(3))
piv = sens.pivot(index="parametro", columns="cambio", values="efecto_%")
piv.plot(kind="barh", color=[CORAL, VERDE], title="Efecto sobre la humedad media (%)"); plt.show()

### 2.4 Dominancia cambiante de bucles (crecimiento logístico)

$\dot P = rP - \frac{r}{K}P^2$: el primer término es el bucle de refuerzo y el segundo el de balance. El bucle que domina es el que tiene mayor efecto sobre la tasa de cambio; el cambio de dominancia ocurre en el punto de inflexión $P = K/2$.

In [ ]:
r, K = 0.2, 1000
t, X = simular(lambda t, x: [r*x[0]*(1 - x[0]/K)], [10], 60, 0.05)
P = X[:, 0]; refuerzo = r*P; balance = r*P**2/K
fig, ax = plt.subplots(1, 2, figsize=(14, 4))
ax[0].plot(t, P, color=AZUL); ax[0].axhline(K/2, color=GRIS, ls=":"); ax[0].set_title("Población")
ax[1].plot(t, refuerzo, color=VERDE, label="aporte del bucle R (rP)"); ax[1].plot(t, balance, color=CORAL, label="aporte del bucle B (rP²/K)")
cambio = t[np.argmax(balance > refuerzo/2)]; ax[1].axvline(t[np.argmax(P >= K/2)], color=GRIS, ls=":", label="inflexión P = K/2"); ax[1].legend(); ax[1].set_title("¿Qué bucle domina?")
plt.show()
print(f"Inflexión simulada en t = {t[np.argmax(P >= K/2)]:.2f}; teórica ln((K−P0)/P0)/r = {math.log((K-10)/10)/r:.2f}")

---
## 3. Tipos de trayectorias

### 3.1 Los seis modos fundamentales de comportamiento
Cada modo nace de una estructura de bucles distinta.

In [ ]:
modelos = {
    "Crecimiento exponencial (R)": (lambda t, x: [0.06*x[0]], [10]),
    "Búsqueda de objetivo (B)": (lambda t, x: [(100 - x[0])/8], [20]),
    "Crecimiento en S (R → B)": (lambda t, x: [0.2*x[0]*(1 - x[0]/100)], [10]),
    # oscilación: inventario con línea de suministro (retardo de 3er orden)
    "Oscilación (B con retardo)": (lambda t, x: [x[3]/1 - 20, max(0, 20 + (100 - x[0])/2) - x[1]/1, x[1]/1 - x[2]/1, x[2]/1 - x[3]/1], [60, 20, 20, 20]),
    # sobrepaso: el límite se percibe con retardo
    "Crecimiento con sobrepaso": (lambda t, x: [0.25*x[0]*(1 - x[1]/100), (x[0] - x[1])/3], [5, 5]),
    # sobrepaso y colapso: el recurso (capacidad) se agota
    "Sobrepaso y colapso": (lambda t, x: [0.3*x[0]*(1 - x[0]/max(x[1]/10, 1e-6)) - 0.02*x[0], -0.6*x[0]], [5, 1500]),
}
fig, axs = plt.subplots(2, 3, figsize=(15, 7)); axs = axs.ravel()
for ax, (nombre, (f, x0)), c in zip(axs, modelos.items(), PAL):
    t, X = simular(f, x0, 50, 0.05); ax.plot(t, X[:, 0], color=c, lw=2.5); ax.set_title(nombre, fontsize=11)
plt.tight_layout(); plt.show()

### 3.2 Equilibrios y estabilidad

En un equilibrio $f(\mathbf{x}^*) = 0$. Linealizamos con la matriz jacobiana $J$ y miramos sus autovalores $\lambda$:

| Autovalores | Tipo |
|---|---|
| reales, todos < 0 | nodo estable |
| reales, alguno > 0 | nodo inestable o silla |
| complejos, Re < 0 | foco estable (espiral hacia adentro) |
| complejos, Re > 0 | foco inestable |
| imaginarios puros | centro |

In [ ]:
def jacobiano(f, x, h=1e-6):
    x = np.asarray(x, float); n = len(x); J = np.zeros((n, n)); f0 = np.asarray(f(0, x), float)
    for j in range(n):
        e = np.zeros(n); e[j] = h; J[:, j] = (np.asarray(f(0, x + e), float) - f0) / h
    return J
def clasificar(J, tol=1e-6):
    lam = np.linalg.eigvals(J); re, im = lam.real, lam.imag
    if np.all(np.abs(im) < tol):
        if np.all(re < -tol): tipo = "nodo estable"
        elif np.all(re > tol): tipo = "nodo inestable"
        elif np.any(re > tol) and np.any(re < -tol): tipo = "punto de silla"
        else: tipo = "no hiperbólico"
    else:
        if np.all(np.abs(re) < tol): tipo = "centro"
        elif np.all(re < 0): tipo = "foco estable"
        else: tipo = "foco inestable"
    return lam, tipo
ejemplos = {"Nodo": [[-1, 0], [0, -2]], "Foco": [[-0.3, -2], [2, -0.3]], "Silla": [[1, 0], [0, -1]], "Centro": [[0, -1], [1, 0]], "Ejercicio 12": [[0, 1], [-4, -1]]}
filas = []
for n, A in ejemplos.items():
    lam, tipo = clasificar(np.array(A, float)); filas.append(dict(sistema=n, autovalores=np.round(lam, 3), tipo=tipo))
pd.DataFrame(filas)

In [ ]:
fig, axs = plt.subplots(1, 4, figsize=(17, 4.2))
Y, Xg = np.mgrid[-2.5:2.5:25j, -2.5:2.5:25j]
for ax, (n, A) in zip(axs, list(ejemplos.items())[:4]):
    A = np.array(A, float); U = A[0, 0]*Xg + A[0, 1]*Y; V = A[1, 0]*Xg + A[1, 1]*Y
    ax.streamplot(Xg, Y, U, V, color=np.hypot(U, V), cmap="viridis", density=1.1, linewidth=1)
    ax.plot(0, 0, "o", color=CORAL); ax.set_title(f"{n}: {clasificar(A)[1]}"); ax.set_aspect("equal")
plt.tight_layout(); plt.show()

### 3.3 Sistemas no lineales: depredador-presa, ciclo límite y caos

In [ ]:
lv = lambda t, x, a=1.0, b=0.1, c=1.5, d=0.075: [a*x[0] - b*x[0]*x[1], -c*x[1] + d*x[0]*x[1]]
vdp = lambda t, x, mu=1.0: [x[1], mu*(1 - x[0]**2)*x[1] - x[0]]
fig, axs = plt.subplots(1, 3, figsize=(17, 4.5))
t, X = simular(lv, [40, 9], 30, 0.01); axs[0].plot(t, X[:, 0], label="presas", color=VERDE); axs[0].plot(t, X[:, 1], label="depredadores", color=CORAL); axs[0].legend(); axs[0].set_title("Lotka-Volterra: oscilación sostenida")
for x0, c in [((0.1, 0), MORADO), ((3.5, 3), CORAL)]:
    t, X = simular(vdp, x0, 30, 0.01); axs[1].plot(X[:, 0], X[:, 1], color=c, label=f"desde {x0}")
axs[1].legend(); axs[1].set_title("Van der Pol: ciclo límite (atractor)")
lor = lambda t, x, s=10, r=28, b=8/3: [s*(x[1]-x[0]), x[0]*(r-x[2]) - x[1], x[0]*x[1] - b*x[2]]
t, A = simular(lor, [1, 1, 1], 30, 0.005); _, B = simular(lor, [1.001, 1, 1], 30, 0.005)
axs[2].plot(t, A[:, 0], color=VERDE, lw=1, label="x(0) = 1.000"); axs[2].plot(t, B[:, 0], color=CORAL, lw=1, label="x(0) = 1.001"); axs[2].legend(); axs[2].set_title("Lorenz: sensibilidad (caos)")
plt.tight_layout(); plt.show()
sep = np.abs(A[:, 0] - B[:, 0]); print(f"Las trayectorias se separan más de 1 unidad a partir de t ≈ {t[np.argmax(sep > 1)]:.1f}")

---
## 4. Bifurcaciones y catástrofes

### 4.1 Diagramas de bifurcación de las formas normales

Para cada μ buscamos numéricamente los equilibrios ($f(x)=0$) y su estabilidad (signo de $f'(x^*)$).

In [ ]:
def equilibrios_1d(f, xs=np.linspace(-3, 3, 6001)):
    y = f(xs); raices = []
    for i in range(len(xs) - 1):
        if y[i] == 0: raices.append(xs[i])
        elif y[i] * y[i+1] < 0: raices.append(brentq(f, xs[i], xs[i+1]))
    return sorted(set(np.round(raices, 6)))
def diagrama(fmu, mus):
    est, ine = [], []
    for mu in mus:
        f = lambda x: fmu(x, mu)
        for x in equilibrios_1d(f):
            d = (f(x + 1e-6) - f(x - 1e-6)) / 2e-6; (est if d < 0 else ine).append((mu, x))
    return np.array(est), np.array(ine)
formas = {"Silla-nodo: μ − x²": lambda x, m: m - x**2, "Transcrítica: μx − x²": lambda x, m: m*x - x**2 + 1e-9, "Horquilla: μx − x³": lambda x, m: m*x - x**3 + 1e-9}
mus = np.linspace(-2, 2, 161)
fig, axs = plt.subplots(1, 3, figsize=(16, 4.2))
for ax, (n, fm) in zip(axs, formas.items()):
    e, i = diagrama(fm, mus)
    if len(e): ax.plot(e[:, 0], e[:, 1], ".", color=VERDE, ms=3, label="estable")
    if len(i): ax.plot(i[:, 0], i[:, 1], ".", color=CORAL, ms=3, label="inestable")
    ax.set_title(n); ax.set_xlabel("μ"); ax.set_ylabel("x*"); ax.legend()
plt.tight_layout(); plt.show()

### 4.2 Bifurcación de Hopf

En coordenadas polares $\dot r = \mu r - r^3$, $\dot\theta = 1$. Para μ < 0 el equilibrio es un foco estable; para μ > 0 aparece un **ciclo límite** de radio $\sqrt{\mu}$.

In [ ]:
def hopf(mu):
    return lambda t, z: [mu*z[0] - z[1] - z[0]*(z[0]**2 + z[1]**2), z[0] + mu*z[1] - z[1]*(z[0]**2 + z[1]**2)]
fig, axs = plt.subplots(1, 3, figsize=(15, 4.6))
for ax, mu in zip(axs, (-0.3, 0.1, 0.5)):
    for z0 in [(1.2, 0), (0.05, 0)]:
        t, Z = simular(hopf(mu), z0, 60, 0.02); ax.plot(Z[:, 0], Z[:, 1], lw=1)
    if mu > 0: th = np.linspace(0, 2*np.pi, 200); ax.plot(np.sqrt(mu)*np.cos(th), np.sqrt(mu)*np.sin(th), "--", color=TINTA, label=f"radio √μ = {np.sqrt(mu):.2f}"); ax.legend()
    ax.set_title(f"μ = {mu}"); ax.set_aspect("equal")
plt.tight_layout(); plt.show()

### 4.3 Duplicación de periodo, caos y exponente de Lyapunov

Mapa logístico $x_{n+1} = r x_n (1 - x_n)$. El exponente de Lyapunov $\lambda = \lim \frac1N\sum \ln|r(1-2x_n)|$ es negativo con orden y positivo con caos.

In [ ]:
rs = np.linspace(2.5, 4.0, 1500); x = 0.5 * np.ones_like(rs); lyap = np.zeros_like(rs)
for _ in range(500): x = rs*x*(1 - x)
pts_r, pts_x = [], []
for k in range(300):
    x = rs*x*(1 - x); lyap += np.log(np.abs(rs*(1 - 2*x)) + 1e-12)
    if k >= 200: pts_r.append(rs.copy()); pts_x.append(x.copy())
lyap /= 300
fig, ax = plt.subplots(2, 1, figsize=(12, 7), sharex=True)
ax[0].plot(np.concatenate(pts_r), np.concatenate(pts_x), ",", color=MORADO, alpha=0.4); ax[0].set_title("Diagrama de bifurcación del mapa logístico")
ax[1].plot(rs, lyap, color=CORAL, lw=1); ax[1].axhline(0, color=TINTA); ax[1].set_title("Exponente de Lyapunov"); ax[1].set_xlabel("r")
for a in ax: a.axvline(3, color=GRIS, ls=":"); a.axvline(3.5699, color=GRIS, ls=":")
plt.show()
print(f"Primer r con caos (λ > 0): {rs[np.argmax(lyap > 0.001)]:.4f}  (Feigenbaum: 3.5699…)")

### 4.4 Catástrofe en cúspide

Potencial $V(x) = \frac{x^4}{4} + \frac{a x^2}{2} + b x$. Equilibrios: $x^3 + a x + b = 0$. Dentro de la región $4a^3 + 27b^2 < 0$ hay **tres** equilibrios (dos estables y uno inestable).

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(17, 4.6))
xs = np.linspace(-2.2, 2.2, 400)
for b, c in zip((-1, 0, 0.6, 1), PAL):
    axs[0].plot(xs, xs**4/4 - xs**2 + b*xs, color=c, label=f"b = {b}")
axs[0].set_title("Potencial con a = −2"); axs[0].legend(); axs[0].set_xlabel("x")
A, B = np.meshgrid(np.linspace(-3, 1, 300), np.linspace(-2.5, 2.5, 300))
n_eq = np.where(4*A**3 + 27*B**2 < 0, 3, 1)
axs[1].contourf(A, B, n_eq, levels=[0, 2, 4], colors=["#E8F8F5", "#FBEFC9"]); aa = np.linspace(-3, 0, 100)
axs[1].plot(aa, np.sqrt(-4*aa**3/27), color=CORAL); axs[1].plot(aa, -np.sqrt(-4*aa**3/27), color=CORAL)
axs[1].text(-2.4, 0, "3 equilibrios\n(biestable)", fontsize=10); axs[1].text(0.1, 1.5, "1 equilibrio", fontsize=10); axs[1].set_xlabel("a"); axs[1].set_ylabel("b"); axs[1].set_title("Conjunto de bifurcación")
# superficie de equilibrio para a = −2 (curva x vs b)
bb = np.linspace(-2, 2, 400); eq = [np.roots([1, 0, -2, b_]) for b_ in bb]
for b_, rts in zip(bb, eq):
    for r_ in rts[np.abs(rts.imag) < 1e-9].real:
        axs[2].plot(b_, r_, ".", ms=2, color=VERDE if 3*r_**2 - 2 > 0 else CORAL)
axs[2].set_title("Equilibrios x* frente a b (a = −2): pliegue"); axs[2].set_xlabel("b"); axs[2].set_ylabel("x*")
plt.tight_layout(); plt.show()

### 4.5 Caso: gusano del abeto (Ludwig, Jones y Holling, 1978) — histéresis

$\dot x = r x (1 - x/k) - \dfrac{x^2}{1 + x^2}$, con $k = 10$. Barremos $r$ hacia arriba y hacia abajo, dejando que el sistema se asiente en cada valor.

In [ ]:
k = 10
gusano = lambda r: (lambda t, x: [r*x[0]*(1 - x[0]/k) - x[0]**2/(1 + x[0]**2)])
def barrido(rs, x0):
    out, x = [], x0
    for r in rs:
        _, X = simular(gusano(r), [x], 150, 0.1); x = X[-1, 0]; out.append(x)
    return np.array(out)
rs = np.linspace(0.3, 0.7, 81); sube = barrido(rs, 0.5); baja = barrido(rs[::-1], 8)[::-1]
fig, ax = plt.subplots(1, 2, figsize=(15, 4.5))
for r in np.linspace(0.3, 0.7, 161):
    f = lambda x: r*x*(1 - x/k) - x**2/(1 + x**2); xs_ = np.linspace(0.001, 12, 6000); y = f(xs_)
    for i in np.where(np.sign(y[:-1]) != np.sign(y[1:]))[0]:
        x_ = brentq(f, xs_[i], xs_[i+1]); d = (f(x_ + 1e-6) - f(x_ - 1e-6)) / 2e-6
        ax[0].plot(r, x_, ".", ms=3, color=VERDE if d < 0 else CORAL)
ax[0].set_title("Equilibrios (verde estable, rojo inestable)"); ax[0].set_xlabel("r")
ax[1].plot(rs, sube, "o-", ms=3, color=CORAL, label="r aumenta"); ax[1].plot(rs, baja, "o-", ms=3, color=MORADO, label="r disminuye"); ax[1].legend(); ax[1].set_title("Histéresis"); ax[1].set_xlabel("r")
plt.show()
salto_up = rs[np.argmax(np.diff(sube) > 2)]; salto_dn = rs[np.argmax(np.diff(baja) > 2)]
print(f"Salto hacia el brote entre r = {salto_up:.3f} y {salto_up + 0.005:.3f}; regreso al estado bajo solo por debajo de r ≈ {salto_dn + 0.005:.3f}")

### 4.6 Señales de alerta temprana

Cerca de una catástrofe el sistema se recupera cada vez más lento de las perturbaciones (**ralentización crítica**): su **varianza** y su **autocorrelación** aumentan antes del salto. Lo vemos agregando ruido y subiendo $r$ poco a poco.

In [ ]:
rng = np.random.default_rng(1); dt = 0.05; T = 3000; n = int(T/dt)
r_t = np.linspace(0.40, 0.60, n); x = 0.6; xs_ = np.empty(n)
for i in range(n):
    r = r_t[i]; x += dt*(r*x*(1 - x/k) - x**2/(1 + x**2)) + 0.03*np.sqrt(dt)*rng.normal(); x = max(x, 0.01); xs_[i] = x
s = pd.Series(xs_[::20], index=r_t[::20]); salto = s.index[np.argmax(s.values > 3)]
pre = s[s.index < salto]; ventana = 150
var = pre.rolling(ventana).var(); ac1 = pre.rolling(ventana).apply(lambda w: pd.Series(w).autocorr(1), raw=True)
fig, ax = plt.subplots(3, 1, figsize=(12, 8), sharex=True)
s.plot(ax=ax[0], color=TINTA, title="Larvas con ruido mientras r sube"); ax[0].axvline(salto, color=CORAL, ls="--")
var.plot(ax=ax[1], color=MORADO, title="Varianza móvil (antes del salto)"); ac1.plot(ax=ax[2], color=VERDE, title="Autocorrelación de orden 1 (antes del salto)")
for a in ax[1:]: a.axvline(salto, color=CORAL, ls="--")
ax[2].set_xlabel("r"); plt.tight_layout(); plt.show()
print(f"Salto observado en r ≈ {salto:.3f}")

---
## 5. Diagramas causales como grafos con signo

Un diagrama causal es un **grafo dirigido** cuyas aristas tienen signo (+ o −). Un bucle es un **ciclo** del grafo y su polaridad es el **producto de los signos**: par número de enlaces negativos → refuerzo (R); impar → balance (B).

In [ ]:
def diagrama_causal(enlaces):
    G = nx.DiGraph()
    for a, b, s in enlaces: G.add_edge(a, b, signo=s)
    return G
def bucles(G):
    filas = []
    for ciclo in nx.simple_cycles(G):
        pares = list(zip(ciclo, ciclo[1:] + ciclo[:1])); negs = sum(G[a][b]["signo"] == "-" for a, b in pares)
        filas.append(dict(bucle=" → ".join(ciclo + ciclo[:1]), enlaces=len(pares), negativos=negs, tipo="R (refuerzo)" if negs % 2 == 0 else "B (balance)"))
    return pd.DataFrame(filas)
def dibujar(G, titulo, pos=None):
    pos = pos or nx.circular_layout(G); fig, ax = plt.subplots(figsize=(8, 6))
    nx.draw_networkx_nodes(G, pos, node_color="white", edgecolors=TINTA, node_size=2600, ax=ax)
    nx.draw_networkx_labels(G, pos, font_size=9, ax=ax)
    for a, b, d in G.edges(data=True):
        nx.draw_networkx_edges(G, pos, edgelist=[(a, b)], edge_color=VERDE if d["signo"] == "+" else CORAL, width=2, arrowsize=18, node_size=2600, connectionstyle="arc3,rad=0.15", ax=ax)
    nx.draw_networkx_edge_labels(G, pos, edge_labels={(a, b): d["signo"] for a, b, d in G.edges(data=True)}, font_color=TINTA, font_size=13, label_pos=0.3, ax=ax)
    ax.set_title(titulo); ax.margins(0.25); ax.axis("off"); plt.show()

epidemia = diagrama_causal([("Susceptibles", "Contagios", "+"), ("Contagios", "Susceptibles", "-"), ("Contagios", "Infectados", "+"), ("Infectados", "Contagios", "+"),
                            ("Infectados", "Recuperaciones", "+"), ("Recuperaciones", "Infectados", "-"), ("Recuperaciones", "Recuperados", "+")])
dibujar(epidemia, "Epidemia: diagrama causal")
bucles(epidemia)

In [ ]:
# Inventario con retardo y ejercicios de polaridad
inventario = diagrama_causal([("Inventario", "Brecha", "-"), ("Inventario deseado", "Brecha", "+"), ("Brecha", "Pedidos", "+"), ("Pedidos", "Llegadas", "+"), ("Llegadas", "Inventario", "+"), ("Ventas", "Inventario", "-")])
display(bucles(inventario))
ej1 = diagrama_causal([("A", "B", "+"), ("B", "C", "-"), ("C", "A", "-")]); ej2 = diagrama_causal([("A", "B", "-"), ("B", "C", "-"), ("C", "A", "-")])
display(bucles(ej1)); display(bucles(ej2))

### 5.1 Los arquetipos son estructuras de bucles

Ejemplo: **límites al crecimiento** (usuarios de una app) y **tragedia de los comunes** (dos pescadores y un banco de peces).

In [ ]:
limites = diagrama_causal([("Usuarios", "Adopciones", "+"), ("Adopciones", "Usuarios", "+"), ("Usuarios", "Mercado restante", "-"), ("Mercado restante", "Adopciones", "+")])
comunes = diagrama_causal([("Captura A", "Ganancia A", "+"), ("Ganancia A", "Esfuerzo A", "+"), ("Esfuerzo A", "Captura A", "+"), ("Captura B", "Ganancia B", "+"), ("Ganancia B", "Esfuerzo B", "+"), ("Esfuerzo B", "Captura B", "+"),
                           ("Captura A", "Peces", "-"), ("Captura B", "Peces", "-"), ("Peces", "Captura A", "+"), ("Peces", "Captura B", "+")])
display(bucles(limites)); display(bucles(comunes))
# simulación de la tragedia de los comunes
def comunes_f(t, x, r=0.3, K=1000, q=0.001, c=0.05, p=1.0):
    N, E1, E2 = x; C1, C2 = q*E1*N, q*E2*N
    return [r*N*(1 - N/K) - C1 - C2, c*(p*C1 - 0.2*E1), c*(p*C2 - 0.2*E2)]
t, X = simular(comunes_f, [1000, 50, 80], 300, 0.1)
fig, ax = plt.subplots(1, 2, figsize=(14, 4)); ax[0].plot(t, X[:, 0], color=AZUL); ax[0].set_title("Peces (recurso común)")
ax[0].axhline(500, color=GRIS, ls=":", label="nivel de máximo rendimiento sostenible (K/2)"); ax[0].legend()
ax[1].plot(t, X[:, 1], label="esfuerzo A", color=VERDE); ax[1].plot(t, X[:, 2], label="esfuerzo B", color=CORAL); ax[1].legend(); ax[1].set_title("Esfuerzo de pesca"); plt.show()
print(f"Peces al final: {X[-1, 0]:.0f} (muy por debajo de K/2 = 500): cada pescador optimiza su bucle R y el recurso común se sobreexplota")

---
## 6. Simbología: funciones de Vensim implementadas en Python

| Vensim | Python aquí |
|---|---|
| `INTEG(flujo, inicial)` | un nivel del vector de estado |
| `STEP(h, t0)`, `PULSE(t0, d)`, `RAMP(p, t0, t1)` | funciones del tiempo |
| `DELAY1`, `DELAY3` | 1 o 3 niveles en serie (retardo material) |
| `SMOOTH` | un nivel que ajusta una percepción (retardo de información) |
| Tabla (lookup) | `np.interp` |

In [ ]:
STEP = lambda h, t0: (lambda t: h * (t >= t0))
PULSE = lambda t0, d: (lambda t: 1.0 * (t0 <= t < t0 + d))
RAMP = lambda p, t0, t1: (lambda t: p * (min(t, t1) - t0) if t >= t0 else 0.0)
entrada = lambda t: 10 + STEP(10, 5)(t)          # la entrada sube de 10 a 20 en t = 5
d = 6
def retardos(t, x):
    d1, s1, s2, s3, sm = x
    return [entrada(t) - d1/d,                                   # DELAY1: un nivel
            entrada(t) - s1/(d/3), s1/(d/3) - s2/(d/3), s2/(d/3) - s3/(d/3),   # DELAY3: tres niveles
            (entrada(t) - sm)/d]                                 # SMOOTH (información)
t, X = simular(retardos, [10*d, 10*d/3, 10*d/3, 10*d/3, 10], 40, 0.02)
fig, ax = plt.subplots()
ax.plot(t, [entrada(tt) for tt in t], color=TINTA, lw=2, label="entrada (STEP en t = 5)")
ax.plot(t, X[:, 0]/d, color=CORAL, label="salida DELAY1"); ax.plot(t, X[:, 3]/(d/3), color=MORADO, label="salida DELAY3"); ax.plot(t, X[:, 4], "--", color=VERDE, label="SMOOTH (percepción)")
ax.legend(); ax.set_title(f"Respuesta de los retardos a un escalón (retardo medio {d})"); plt.show()

In [ ]:
# Tabla (lookup): efecto no lineal de la densidad sobre la natalidad
dens_x = [0, 0.5, 1.0, 1.5, 2.0]; efecto_y = [1.2, 1.1, 1.0, 0.6, 0.2]
efecto = lambda dens: np.interp(dens, dens_x, efecto_y)
pob = lambda t, x: [0.05 * efecto(x[0]/500) * x[0] - x[0]/40]
t, X = simular(pob, [50], 200, 0.1)
fig, ax = plt.subplots(1, 2, figsize=(14, 4)); xx = np.linspace(0, 2, 100); ax[0].plot(xx, efecto(xx), color=MORADO); ax[0].plot(dens_x, efecto_y, "o", color=CORAL); ax[0].set_title("Tabla: efecto de la densidad"); ax[0].set_xlabel("densidad relativa")
ax[1].plot(t, X[:, 0], color=AZUL); ax[1].set_title("Población con efecto no lineal"); plt.show()
print(f"Equilibrio: natalidad × efecto = 1/40 → efecto = {1/40/0.05:.2f} → densidad ≈ {np.interp(0.5, efecto_y[::-1], dens_x[::-1]):.2f} → población ≈ {500*np.interp(0.5, efecto_y[::-1], dens_x[::-1]):.0f}; simulado {X[-1, 0]:.0f}")

---
## 7. Etapas para elaborar un modelo: epidemia en el campus

### Etapa 1 · Definición del problema
Un virus respiratorio llega a un campus de **5 000** estudiantes. Pregunta: ¿cuándo será el pico, cuántos se enfermarán y qué medida lo reduce más? Horizonte: 120 días. **Modo de referencia**: curva en campana de casos activos.

### Etapa 2 · Conceptualización (hipótesis dinámica)
Ya vimos el diagrama causal con un bucle de contagio (R) y dos de balance (B) en la sección 5.

### Etapa 3 · Formalización

In [ ]:
N = 5000
def sir_modelo(beta=0.35, duracion=10, vacunados=0, cuarentena=0.0):
    return ModeloDS(
        niveles={"S": (N - 5 - vacunados, [], ["contagios"]), "I": (5, ["contagios"], ["recuperaciones"]), "R": (vacunados, ["recuperaciones"], [])},
        flujos={"contagios": lambda v: v["beta"] * (1 - v["cuarentena"]) * v["S"] * v["I"] / N, "recuperaciones": lambda v: v["I"] / v["duracion"]},
        constantes={"beta": beta, "duracion": duracion, "cuarentena": cuarentena})
m = sir_modelo(); display(m.estructura())
print("Ecuaciones:\n S = INTEG(-contagios, 4995)\n I = INTEG(contagios - recuperaciones, 5)\n R = INTEG(recuperaciones, 0)\n contagios = beta·S·I/N   [personas/día]\n recuperaciones = I/duracion   [personas/día]")

### Etapa 4 · Comportamiento

In [ ]:
df = m.correr(120, dt=0.1, metodo="rk4")
pico = df["I"].idxmax()
fig, ax = plt.subplots(); df[["S", "I", "R"]].plot(ax=ax, color=[AZUL, CORAL, VERDE], lw=2.5); ax.axvline(pico, color=GRIS, ls=":"); ax.set_title("Modelo SIR del campus"); plt.show()
print(f"R0 = {0.35*10:.1f} · pico de {df['I'].max():,.0f} infectados en el día {pico:.1f} · total enfermos {df['R'].iloc[-1]:,.0f}")

### Etapa 5 · Evaluación (validación)

In [ ]:
pruebas = []
d0 = sir_modelo().correr(120, 0.1, "rk4")
pruebas.append(("Conservación S + I + R = N", np.allclose(d0[["S", "I", "R"]].sum(axis=1), N)))
dz = ModeloDS({"S": (N, [], ["contagios"]), "I": (0, ["contagios"], ["recuperaciones"]), "R": (0, ["recuperaciones"], [])}, m.flujos, constantes=m.const).correr(120, 0.1, "rk4")
pruebas.append(("Condición extrema I(0) = 0 → nadie se enferma", dz["I"].max() < 1e-9))
pruebas.append(("Condición extrema β = 0 → no hay contagios", sir_modelo(beta=0).correr(120, 0.1, "rk4")["contagios"].max() < 1e-9))
pruebas.append(("Ningún nivel es negativo", (d0[["S", "I", "R"]] >= -1e-9).all().all()))
dh = sir_modelo().correr(120, 0.05, "rk4")
pruebas.append(("Reducir DT a la mitad no cambia el pico (< 0.5 %)", abs(dh["I"].max() - d0["I"].max()) / d0["I"].max() < 0.005))
display(pd.DataFrame(pruebas, columns=["prueba", "superada"]))
# sensibilidad Monte Carlo sobre β y duración
rng = np.random.default_rng(3); filas = []
for _ in range(200):
    b, dur = rng.uniform(0.25, 0.45), rng.uniform(7, 13); d_ = sir_modelo(b, dur).correr(120, 0.25, "rk4")
    filas.append(dict(beta=b, duracion=dur, R0=b*dur, pico=d_["I"].max(), dia_pico=d_["I"].idxmax()))
mc = pd.DataFrame(filas)
fig, ax = plt.subplots(1, 2, figsize=(14, 4)); sns.scatterplot(data=mc, x="R0", y="pico", hue="duracion", palette="viridis", ax=ax[0]); ax[0].set_title("Sensibilidad: pico frente a R0")
sns.histplot(mc["dia_pico"], bins=25, color=MORADO, ax=ax[1]); ax[1].set_title("Incertidumbre en el día del pico"); plt.show()
print(mc[["pico", "dia_pico"]].describe().round(1).to_string())

### Etapa 6 · Explotación: diseño de políticas

In [ ]:
politicas = {"Sin medidas": dict(), "Tapabocas y aforos (β = 0.20)": dict(beta=0.20), "Cuarentena del 30 % de contactos": dict(cuarentena=0.30),
             "Vacunar a 2 000 antes": dict(vacunados=2000), "Combinada (β = 0.25 + 1 000 vacunados)": dict(beta=0.25, vacunados=1000)}
fig, ax = plt.subplots(); filas = []
for (n, kw), c in zip(politicas.items(), PAL):
    d_ = sir_modelo(**kw).correr(150, 0.1, "rk4"); ax.plot(d_.index, d_["I"], color=c, lw=2, label=n)
    filas.append(dict(politica=n, pico=d_["I"].max(), dia_pico=d_["I"].idxmax(), enfermos=d_["R"].iloc[-1] - kw.get("vacunados", 0)))
ax.legend(fontsize=9); ax.set_title("Infectados activos según la política"); plt.show()
pd.DataFrame(filas).round(0)

**Conclusión de la etapa:** las medidas que reducen el contacto (β) bajan el pico y lo retrasan («aplanar la curva»); la vacunación previa reduce el número de susceptibles y con él el total de enfermos. El umbral de inmunidad colectiva es $1 - 1/R_0$.

---
## 8. Los 18 ejercicios de la presentación, resueltos

In [ ]:
print("1)", bucles(diagrama_causal([("A", "B", "+"), ("B", "C", "-"), ("C", "A", "-")]))["tipo"][0])
print("2)", bucles(diagrama_causal([("A", "B", "-"), ("B", "C", "-"), ("C", "A", "-")]))["tipo"][0], "→ búsqueda de objetivo")
print("3) Saldo = nivel; depósitos, intereses y retiros = flujos; tasa de interés = constante")
print("4)", bucles(diagrama_causal([("Usuarios", "Contenido", "+"), ("Contenido", "Atractivo", "+"), ("Atractivo", "Usuarios", "+")]))["tipo"][0])
print("5) Límites al crecimiento")
print("6)", bucles(diagrama_causal([("Temperatura", "Brecha", "-"), ("Brecha", "Giro de la llave", "+"), ("Giro de la llave", "Temperatura", "+")]))["tipo"][0], "con retardo → oscila")
print(f"7) Duplicación: ln2/0.07 = {math.log(2)/0.07:.2f} años")
print(f"8) T(10) = 20 + 70·e^-1 = {20 + 70*math.exp(-1):.2f} °C")
t, X = simular(lambda t, x: [0.05*x[0]], [100], 3, 1, "euler"); print(f"9) Euler: {np.round(X[:, 0], 2).tolist()} · exacto {100*math.exp(0.15):.2f}")
print(f"10) t* = ln(99)/0.2 = {math.log(99)/0.2:.2f}")
for xe in (0, 3): print(f"11) x* = {xe}: f'(x*) = {3 - 2*xe} →", "estable" if 3 - 2*xe < 0 else "inestable")
print("12)", clasificar(np.array([[0, 1], [-4, -1]], float)))
for xe in (2, -2): print(f"13) x* = {xe}: f'(x*) = {-2*xe} →", "estable" if -2*xe < 0 else "inestable")
x = 0.3
for _ in range(1000): x = 2.8*x*(1-x)
print(f"14) r = 2.8 → x* = {x:.4f} (1 − 1/r = {1 - 1/2.8:.4f})", end="; ")
x = 0.3; orb = []
for i in range(1100):
    x = 3.2*x*(1-x)
    if i > 1090: orb.append(round(x, 4))
print("r = 3.2 → órbita", sorted(set(orb)))
rts = np.roots([1, 0, -3, 0]); print("15) equilibrios", np.round(rts.real, 4), "→ estables donde 3x² − 3 > 0:", [float(round(r_, 4)) for r_ in rts.real if 3*r_**2 - 3 > 0])
print(f"16) R0 = 3.0 · umbral = {1 - 1/3:.3f}")
print("17) 6/3 = 2 días por etapa")
print("18) Definición del problema → conceptualización → formalización → comportamiento → evaluación → explotación")

---
### Para seguir practicando
1. Cambia `t_riego` del invernadero de 2 a 0.5 días: ¿qué pasa con el error en estado estacionario y con el consumo de agua?
2. Agrega al modelo SIR un flujo de pérdida de inmunidad (R → S) con duración 90 días. ¿Qué modo de comportamiento aparece?
3. En el gusano del abeto, ¿qué valor de k elimina la histéresis?
4. Usa `bucles()` para analizar un diagrama causal de tu propio proyecto.